In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pvlib
from pvlib import clearsky, atmosphere, solarposition
from pvlib.iotools import read_tmy3
from pvlib.pvsystem import PVSystem, FixedMount, Array
from pvlib.location import Location
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS
from pvlib.location import Location
import datetime
import pytz
import os
import itertools
import inspect
import calendar
import h5py
from pvlib import pvsystem, location, modelchain, iotools,tools
import pathlib
from dataclasses import dataclass
from pvlib import tracking
from netCDF4 import num2date
from requests.exceptions import HTTPError
from xml.etree.ElementTree import ParseError
from pvlib.irradiance import campbell_norman, get_extra_radiation, disc, louche, erbs_driesse, ghi_from_poa_driesse_2023
from pvlib.irradiance import _liujordan, get_total_irradiance
from siphon.catalog import TDSCatalog
from siphon.ncss import NCSS
import warnings
from pvlib._deprecation import deprecated
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.optimize import curve_fit
from scipy.optimize import least_squares
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.optimize import curve_fit
from scipy.optimize import least_squares

In [2]:
#BSRN QC
data_path ='STEL_2015_201620182019C_GHI.csv'
df = pd.read_csv(data_path, index_col=[0], parse_dates=[0])
#df.index = df.index.tz_localize('Africa/Johannesburg')  # Make the index timezone aware
original_entries = df.shape[0]
df.head(2)  # Print the first two lines of the DataFrame

,ghi_measured,k_t,zenith,airmass,ast,extraterrestrial_radiation
DATE,,,,,,
2015-01-01 07:31:00+02:00,333.224,512.188103,68.777384,2.746518,0.361993,1414.91335
2015-01-01 07:32:00+02:00,338.062,516.803982,68.576732,2.722327,0.365255,1414.91335


In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

# Clean the DataFrame: Replace infinite values with NaNs, then drop any rows with NaNs.
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# Define the model function using 'zenith' and 'extraterrestrial_radiation'
def model_function(xdata, a, b):
    # xdata now contains 'zenith' and 'extraterrestrial_radiation'
    zenith, extraterrestrial_radiation = xdata
    # Convert zenith from degrees to radians for the cosine function
    return a * extraterrestrial_radiation * np.cos(np.deg2rad(zenith))**b

# Prepare the independent variables from the cleaned DataFrame
zenith = df['zenith'].to_numpy()  # using the 'zenith' column (angle in degrees)
extraterrestrial_radiation = df['extraterrestrial_radiation'].to_numpy()

# Stack the variables into a 2-row array for curve_fit
xdata = np.vstack((zenith, extraterrestrial_radiation))

# The measured GHI values are our target variable from the cleaned DataFrame
ydata = df['ghi_measured'].to_numpy()

# Initial guesses for the parameters a and b
initial_guess = [0.81, 1.15]

# Perform curve fitting to calibrate the parameters
popt, pcov = curve_fit(model_function, xdata, ydata, p0=initial_guess)

# Extract and display the calibrated parameters
a_calibrated, b_calibrated = popt
print("Calibrated parameters:")
print(f"a (scaling factor) = {a_calibrated}")
print(f"b (exponent)       = {b_calibrated}")



Calibrated parameters:
a (scaling factor) = 0.7899001768097357
b (exponent)       = 1.1224064504200844
